In [1]:
from pathlib import Path

import torch
import ultralytics
from ultralytics import YOLO

In [2]:
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
Ultralytics: 8.4.124
CUDA available: True
GPU: NVIDIA GeForce RTX 5050 Laptop GPU


In [5]:
PROJECT_ROOT = Path(
    r"D:\master\Master_Drone_Detection\02_datasets\DUT_Anti_UAV"
)

DATA_YAML = (
    PROJECT_ROOT
    / "data.yaml"
)

EXP3_PROJECT = (
    PROJECT_ROOT
    / "runs"
    / "04_experiments"
)

EXP3_NAME = "EXP003_YOLOv8s_P2_960"

print("Dataset YAML:")
print(DATA_YAML)

print("\nExperiment directory:")
print(EXP3_PROJECT / EXP3_NAME)

assert DATA_YAML.exists(), "data.yaml not found!"

Dataset YAML:
D:\master\Master_Drone_Detection\02_datasets\DUT_Anti_UAV\data.yaml

Experiment directory:
D:\master\Master_Drone_Detection\02_datasets\DUT_Anti_UAV\runs\04_experiments\EXP003_YOLOv8s_P2_960


add P2 layer for upsampling

In [6]:
model_p2 = YOLO(
    "yolov8s-p2.yaml"
)

In [8]:
model_p2.info()
print(model_p2.model)

YOLOv8s-p2 summary: 161 layers, 10,884,336 parameters, 10,884,320 gradients, 39.7 GFLOPs


DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C2f(
      (cv1): Conv(
        (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(96, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
    

In [9]:
print(
    "Detection strides:",
    model_p2.model.stride
)

Detection strides: tensor([ 4.,  8., 16., 32.])


In [10]:
model_p2.load(
    "yolov8s.pt"
)

Transferred 219/437 items from pretrained weights


YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(96, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_s

In [11]:
model_p2.info()

YOLOv8s-p2 summary: 161 layers, 10,884,336 parameters, 10,884,320 gradients, 39.7 GFLOPs


(161, 10884336, 10884320, 39.6701184)

In [12]:
print(
    "Strides after loading weights:",
    model_p2.model.stride
)

Strides after loading weights: tensor([ 4.,  8., 16., 32.])


In [13]:
model_p2.info(
    detailed=False,
    verbose=True
)

YOLOv8s-p2 summary: 161 layers, 10,884,336 parameters, 10,884,320 gradients, 39.7 GFLOPs


(161, 10884336, 10884320, 39.6701184)

In [14]:
results_exp3 = model_p2.train(

    data=str(DATA_YAML),

    epochs=100,

    imgsz=960,

    batch=8,

    patience=20,

    workers=4,

    device=0,

    project=str(EXP3_PROJECT),

    name=EXP3_NAME
)

New https://pypi.org/project/ultralytics/8.4.154 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.124  Python-3.11.0 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5050 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\master\Master_Drone_Detection\02_datasets\DUT_Anti_UAV\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0

In [16]:
from ultralytics import YOLO

model = YOLO(
r"D:\master\Master_Drone_Detection\02_datasets\DUT_Anti_UAV\runs\04_experiments\EXP003_YOLOv8s_P2_960\weights\best.pt"
)


results = model.predict(
    source=r"D:\master\Master_Drone_Detection\02_datasets\DUT_Anti_UAV\images\val",
    imgsz=960,
    conf=0.25,
    device=0,
    save=False
)


WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/2600 D:\master\Master_Drone_Detection\02_datasets\DUT_Anti_UAV\images\val\00001.jpg: 576x960 1 UAV, 70.4ms
image 2/2600 D:\master\Master_Drone_Detection\02_datasets\DUT_Anti_UAV\images\val\00002.jpg: 544x960 1 UAV, 44.9ms
image 3/2600 D:\master\Master_Drone_Detection\02_datasets\DUT_Anti_UAV\images\val\00003.jpg: 704x960 1 UAV, 42.6ms
image 4/2600 D:\master\Master_Drone_Detection\02_datasets\DUT_Anti_UAV\images\val\00004.jpg: 640x960 1 UAV, 43.9ms
i

In [ ]:
from ultralytics import YOLO

model = YOLO(
r"D:\master\Master_Drone_Detection\02_datasets\DUT_Anti_UAV\runs\04_experiments\EXP003_YOLOv8s_P2_960\weights\best.pt"
)
    
metrics = model.val(
    data="D:/master/Master_Drone_Detection/02_datasets\DUT_Anti_UAV/data.yaml",
    imgsz=960,
    device=0
)

print(metrics)

Ultralytics 8.4.124  Python-3.11.0 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5050 Laptop GPU, 8151MiB)
YOLOv8s-p2 summary (fused): 91 layers, 10,626,708 parameters, 0 gradients, 36.6 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 319.8110.6 MB/s, size: 190.2 KB)
val: Scanning D:\master\Master_Drone_Detection\02_datasets\DUT_Anti_UAV\labels\val.cache... 2600 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2600/2600  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 163/163 3.2it/s 50.5s0.3ss
                   all       2600       2621      0.974       0.92      0.949      0.661
Speed: 3.0ms preprocess, 12.9ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to D:\master\Master_Drone_Detection\notebooks\runs\detect\val-12
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.met

In [20]:
from ultralytics import YOLO

model = YOLO(
r"D:\master\Master_Drone_Detection\02_datasets\DUT_Anti_UAV\runs\04_experiments\EXP003_YOLOv8s_P2_960\weights\best.pt"
)


results = model.predict(
    source=r"D:\master\Master_Drone_Detection\02_datasets\DUT_Anti_UAV\images\val",
    imgsz=960,
    conf=0.25,
    device=0,
    save=False
)


WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/2600 D:\master\Master_Drone_Detection\02_datasets\DUT_Anti_UAV\images\val\00001.jpg: 576x960 1 UAV, 40.2ms
image 2/2600 D:\master\Master_Drone_Detection\02_datasets\DUT_Anti_UAV\images\val\00002.jpg: 544x960 1 UAV, 44.8ms
image 3/2600 D:\master\Master_Drone_Detection\02_datasets\DUT_Anti_UAV\images\val\00003.jpg: 704x960 1 UAV, 59.9ms
image 4/2600 D:\master\Master_Drone_Detection\02_datasets\DUT_Anti_UAV\images\val\00004.jpg: 640x960 1 UAV, 50.6ms
i